<a href="https://colab.research.google.com/github/npeyton/ml_materials/blob/main/karpathy_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-11 13:43:23--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-11-11 13:43:23 (21.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder - take string output list of integers.
decode = lambda l: ''.join([itos[i] for i in l]) #decoder: take a list of integers, output a string


In [ ]:
#importing pytorch and tokenizing the entire dataset.
import torch
data = torch.tensor(encode(text),dtype= torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [ ]:
# training and validation split
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
#block sizing
block_size= 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
# so the inputs are the block size characters, and Y is the next character
# Also allowing for there to be predictions even if there is are not 8 input characters
#allows for prediction with just 1 character which is important for model use
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range (block_size):
  context = x[:t+1]
  target = y[t]
  print(f'When input is {context} the target is:{target}')

When input is tensor([18]) the target is:47
When input is tensor([18, 47]) the target is:56
When input is tensor([18, 47, 56]) the target is:57
When input is tensor([18, 47, 56, 57]) the target is:58
When input is tensor([18, 47, 56, 57, 58]) the target is:1
When input is tensor([18, 47, 56, 57, 58,  1]) the target is:15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is:47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is:58


In [ ]:
# batching
torch.manual_seed(1337)
batch_size = 4 # how many sequences will we process in parallel (ie x y pairs)
block_size = 8 # what is the max context length for a prediction.

def get_batch(split):
  #generate a small batch of x an Y
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('-----')

for b in range(batch_size): # batch dimension
  for t in range (block_size): # time dimensions
    context = xb[b,: t+1]
    target = yb[b,t]
    print(f'when input is {context.tolist()} the target is {target}')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target is 43
when input is [24, 43] the target is 58
when input is [24, 43, 58] the target is 5
when input is [24, 43, 58, 5] the target is 57
when input is [24, 43, 58, 5, 57] the target is 1
when input is [24, 43, 58, 5, 57, 1] the target is 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
when input is [44] the target is 53
when input is [44, 53] the target is 56
when input is [44, 53, 56] the target is 1
when input is [44, 53, 56, 1] the target is 58
when input is [44, 53, 56, 1, 58] the target 

In [ ]:
# importing dependencies - going with a bigram language model.
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [ ]:
class BigramLanguageModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    #each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self, idx, targets=None):
    # idx and targets are both (B,T) tensor of integers
    logits = self.token_embedding_table(idx) #( Batch, Time , Channel)
    # so basically just returning the logits when passed in a single token.

    # if we have targets we will calculate loss, if not no need to calculate (will just return logits).
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits =  logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits,targets)

    return logits, loss


  def generate(self, idx, max_new_tokens):
    # takes some idx (which is the context) and outputs (B,T+1) over again for new tokens
    # idx aka some input is (B,T) array of indices in the current context and we want to
    # output some amount of new tokens given the logits from the context
    for _ in range (max_new_tokens):
      # get the predictions
      logits, loss = self(idx)
      #focus only on the last time step
      logits = logits [:,-1, :] # becomes (B,C)
      #apply softmax to get some probabilities
      probs = F.softmax(logits,dim=-1) # (B,C)
      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples =1)
      # append sampled index to the runnning sequence
      idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
    return idx


m = BigramLanguageModel(vocab_size)
logits,loss = m(xb,yb)
print( logits.shape)

idx = torch.zeros((1,1),dtype=torch.long)

print(decode(m.generate(idx = torch.zeros((1,1),dtype=torch.long) ,max_new_tokens = 100)[0].tolist()))

torch.Size([32, 65])

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
# create a Pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [ ]:
# training the model

batch_size = 32

for steps in range (10000):

  #sample a batch of data
  xb, yb = get_batch('train')

  #evaluate the loss
  logits, loss = m(xb,yb)
  # zero out the gradients
  optimizer.zero_grad(set_to_none=True)
  # get the gradients
  loss.backward()
  # take a step down the gradient.
  optimizer.step()

print(loss.item())

2.382369041442871


Ok so below is the bigram model trained to a loss of 2.5 ish and its ability to predict. Not very good but certainly interesting.

In [ ]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype=torch.long) ,max_new_tokens = 100)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulsee


SELF ATTENTION!!!!!

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2 #batch, time, channels
x = torch.randn(B,T,C)
x.size()


torch.Size([4, 8, 2])

In [ ]:
# we want to introduce the idea of attention / let the tokens talk to each other.
# aka we want the logits for the next token summed across the context.
# We want x[b,t] = mean_ {i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] #(t,C)
    xbow [b,t] = torch.mean(xprev,0)


In [ ]:
# the above is good but it is inefficient. version 2
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1,keepdim= True)
xbow2 = wei @ x # (T,T) @ (B,T,C) --> (B,T,C)
torch.allclose(xbow2,xbow3)


True

In [ ]:
# version 3
tril = torch.tril(torch.ones(T,T))
#create the matrix full of zeros
wei = torch.zeros((T,T))
# then fill the zeros with negative infinity
wei = wei.masked_fill(tril == 0, float('-inf'))
# then take softmax of that matrix.
wei = F.softmax(wei,dim=-1)
# then we multiply the matrix with the softmax triangle and get the weights
xbow3 = wei @ x
#so basically we are taking a weighted average of all of the past tokens to
# inform the prediction of the next token.
# what this does is essentially makes a matrix of how much differnet tokens
# are affected by each other or their "affinity"
# this is self attention.
torch.allclose(xbow2,xbow3)

True

In [ ]:
# version 4
B,T,C = 4,8,32 #batch, time, channels
x = torch.randn(B,T,C)

# single head of self attention
head_size = 16
key = nn.Linear(C, head_size, bias = False)
query = nn.Linear(C, head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)
wei = q @ k.transpose(-2,-1)


tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei,dim=-1)
out = wei @ x

v = value(x)
out = wei @ v
out.shape

torch.Size([4, 8, 16])

Ok so for the sentence "I deposited money at the bank" here is how it would work.

ok so the bank query is looking for things that would describe itself loosely interperetable to looking for something financial or something river related in the key vector. They key for both "deposit" and for "money" map decently well onto that vector and so they would be activated more and the words "it" and "at" less so. Then the result would be that they would get the highest attention scores and the resulting bank vectyor would have high activations coming from the values from deposit and money. So the information from money and deposit would now be embedded into the bank vector.

